<a href="https://colab.research.google.com/github/YashThirani21/docagent/blob/main/notebooks/colab_notebook_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# docagent on Google Colab (vLLM), from GitHub

Run top to bottom on a GPU runtime (Runtime → Change runtime type → GPU). The code is cloned from
[github.com/YashThirani21/docagent](https://github.com/YashThirani21/docagent). **No setup or tokens needed.**

Where state that outlives the runtime (`store/` and the run outputs) goes is picked in cell 1 by `PERSIST`:
- `"auto"` (default): `"hf"` if this Colab account has an `HF_TOKEN` secret, otherwise `"none"`.
- `"none"`: nothing is saved between sessions. The index is built fresh, and the run outputs are downloaded to your
  computer as a zip at the end.
- `"drive"`: your own `MyDrive/ColabProjects/` (asks for Drive access once).
- `"hf"`: a private Hugging Face dataset repo, `<your HF username>/docagent-store`, in the account of whoever runs the
  notebook. Every save is a commit, so earlier stores and runs stay available. Needs a Colab secret `HF_TOKEN`
  (🔑 icon in the left bar) holding a token with write access.

## 1. Settings

In [1]:
import os, time
from pathlib import Path

REPO_URL = "https://github.com/YashThirani21/docagent.git"
REF = "main"
PROJECT = Path("/content/senior-data-scientist")

PERSIST = "auto"                # "auto", "none" (session only), "drive" (your MyDrive/ColabProjects) or "hf"
HF_DATASET = None               # for "hf" only: <token owner>/docagent-store, created private on the first save
DRIVE_DIR = Path("/content/drive/MyDrive/ColabProjects")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")             # run outputs are saved under runs/<RUN_ID>/

if PERSIST in ("auto", "hf"):
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        from huggingface_hub import whoami
        # Each person saves to their own account: anyone else's token can't write to the maintainer's repo
        HF_DATASET = HF_DATASET or f"{whoami()['name']}/docagent-store"
        PERSIST = "hf"
    except Exception as e:      # no such secret, notebook access not granted, or the token is invalid
        if PERSIST == "hf":
            raise RuntimeError(
                f"No usable Colab secret HF_TOKEN ({type(e).__name__}). Add a token with Write access in "
                "Colab's 🔑 Secrets panel as HF_TOKEN with 'Notebook access' on, or set PERSIST = \"none\"."
            ) from None
        PERSIST = "none"
if PERSIST == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
elif PERSIST not in ("hf", "none"):
    raise ValueError(f"PERSIST must be 'auto', 'none', 'drive' or 'hf', not {PERSIST!r}")
print(f"persist: {PERSIST}" + (f" ({HF_DATASET})" if PERSIST == "hf" else "") + f", run: {RUN_ID}")

persist: none, run: 20261021-161524


## 2. Clone the project

In [2]:
import sys
if "chat" in sys.modules:
    # The running chat app keeps store/traces.db open: deleting it now sends its traces to a deleted file
    raise RuntimeError("The chat app is running from this project. Runtime → Restart session, then run from cell 1.")
!rm -rf {PROJECT}
!git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git checkout -q {REF} && git log -1 --format="running %h  %s  (%cd)" --date=short

/content/senior-data-scientist
running 51820e9  Capping retries to fix p95 latency  (2026-10-21)


## 3. Save / restore helpers

Every read and write of saved state goes through these three functions (no-ops or local downloads when
`PERSIST` is `"none"`), so switching `PERSIST` changes nothing
else in the notebook. Hugging Face calls run through the `hf` CLI with `HF_HUB_OFFLINE=0`, because the docagent
config puts the kernel in offline mode once it is loaded (`privacy.offline`).

In [3]:
import shutil, sqlite3, subprocess

STORE_ARCHIVE = Path("/content/store-backup.tar.gz")
RUN_ITEMS = ["outputs", "eval/results", "store/logs", "vllm.log"]   # + store/traces.db, snapshotted below
TRACES_DB = Path("store/traces.db")

def _hf(*args):
    env = {**os.environ, "HF_HUB_OFFLINE": "0"}
    return subprocess.run(["hf", *args, "--repo-type", "dataset"], env=env, capture_output=True, text=True)

def pull_store():
    """Restore store/ from the last saved backup; on the first run there is none."""
    STORE_ARCHIVE.unlink(missing_ok=True)
    if PERSIST == "none":
        print("Session-only run: starting with an empty store; cell 10 builds the index.")
        return
    if PERSIST == "hf":
        # Fresh folder: if the Hub can't be reached, `hf download` silently returns a file already in --local-dir
        pull_dir = Path("/content/hf_pull")
        shutil.rmtree(pull_dir, ignore_errors=True)
        r = _hf("download", HF_DATASET, STORE_ARCHIVE.name, "--local-dir", str(pull_dir))
        found = r.returncode == 0 and (pull_dir / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.move(pull_dir / STORE_ARCHIVE.name, STORE_ARCHIVE)
        error = next((l for l in r.stderr.splitlines() if l.startswith("Error")), r.stderr.strip()[:200])
        missing = f"{HF_DATASET}/{STORE_ARCHIVE.name} ({error})"
    else:
        found = (DRIVE_DIR / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.copy2(DRIVE_DIR / STORE_ARCHIVE.name, STORE_ARCHIVE)
        missing = DRIVE_DIR / STORE_ARCHIVE.name
    if not found:
        # First run: cell 10 builds the index from scratch, cell 12 saves the backup.
        print(f"No backup at {missing} - starting with an empty store (first run).")
        return
    subprocess.run(["tar", "-xzf", str(STORE_ARCHIVE), "-C", str(PROJECT)], check=True)
    print("Store restored successfully.")

def push_store():
    """Save store/ as one archive; on HF each save is a new commit of the same file."""
    if PERSIST == "none":
        print("Session-only run: store/ is not saved (it is rebuilt by cell 10 next time).")
        return
    if (PROJECT / TRACES_DB).is_file():
        # traces.db is in WAL mode: recent requests sit in traces.db-wal until checkpointed into the main file
        with sqlite3.connect(PROJECT / TRACES_DB) as con:
            con.execute("PRAGMA wal_checkpoint(FULL)")
    subprocess.run(["tar", "-czf", str(STORE_ARCHIVE), "-C", str(PROJECT), "store"], check=True)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(STORE_ARCHIVE), STORE_ARCHIVE.name, "--private",
                "--commit-message", f"store backup {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        where = f"hf.co/datasets/{HF_DATASET}"
    else:
        shutil.copy2(STORE_ARCHIVE, DRIVE_DIR / STORE_ARCHIVE.name)
        where = DRIVE_DIR / STORE_ARCHIVE.name
    print(f"Backup saved: {where}")
    print(f"Archive size: {STORE_ARCHIVE.stat().st_size / (1024**3):.2f} GB")

def push_run():
    """Save outputs, eval results, traces and logs under runs/<RUN_ID>/."""
    staging = Path("/content/run_out") / RUN_ID
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    for item in RUN_ITEMS:
        p = PROJECT / item
        if p.is_dir():
            shutil.copytree(p, staging / p.name)
        elif p.is_file():
            shutil.copy2(p, staging / p.name)
        else:
            print(f"skipped (not found): {item}")
    if (PROJECT / TRACES_DB).is_file():
        # A copy of the file alone misses what is still in traces.db-wal; the backup API includes it
        with sqlite3.connect(PROJECT / TRACES_DB) as src, sqlite3.connect(staging / TRACES_DB.name) as dst:
            src.backup(dst)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(staging), f"runs/{RUN_ID}", "--private",
                "--commit-message", f"run {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        print(f"Run saved: hf.co/datasets/{HF_DATASET}/tree/main/runs/{RUN_ID}")
    elif PERSIST == "none":
        from google.colab import files
        zip_path = shutil.make_archive(f"/content/docagent-run-{RUN_ID}", "zip", staging)
        files.download(zip_path)
        print(f"Run downloaded to your computer: {Path(zip_path).name}")
    else:
        shutil.copytree(staging, DRIVE_DIR / "run_out" / RUN_ID, dirs_exist_ok=True)
        print(f"Run saved: {DRIVE_DIR / 'run_out' / RUN_ID}")

## 4. Restore `store/` from the last backup (skipped on the first run)

In [4]:
pull_store()

Session-only run: starting with an empty store; cell 10 builds the index.


## 5. Install vLLM and the docagent package

In [5]:
%cd /content/senior-data-scientist

!pip install -q vllm==0.31.0 -e . -c requirements-gpu.txt

import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import docagent, vllm; print('docagent and vllm', vllm.__version__, 'installed')"],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Install failed; rerun this cell without -q to see pip's error.\n" + r.stderr[-1500:])
print(r.stdout.strip())

/content/senior-data-scientist
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

## 6. Check the GPU

In [6]:
import shutil, subprocess
# vLLM and the bge-m3 embedder (device: cuda in the colab configs) both need a GPU; stop before cells 7-10 fail
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode:
    raise RuntimeError("No GPU in this session. Runtime → Change runtime type → T4 GPU (or L4/A100), "
                       "then Runtime → Run all. If no GPU is offered, Colab's free GPU quota is used up for now.")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA L4, 23034 MiB, 0 MiB


## 7. Pick the config and start the matching vLLM server

In [7]:
import os, yaml
%cd /content/senior-data-scientist

# Pick the agent version here; the vLLM server is started to match it.
CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml"
# CONFIG = "configs/colab_vllm.yaml"
MAX_LEN = 16384                          # vLLM --max-model-len, as in the configs (results up to 2026-10-19 ran at 8192)
GPU_UTIL = 0.90                          # share of GPU memory for vLLM; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG   # CLI, chat app and Agent() all read this
SERVED = yaml.safe_load(open(CONFIG))["llm"]["model"]   # vLLM must serve exactly the name the config asks for

print(SERVED)
if SERVED.endswith("-awq"):
    # AWQ 4-bit weights in fp16: fits a T4
    print(f"{CONFIG}: serving AWQ as {SERVED}")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
else:
    # start_vllm.sh picks the weights for the GPU: bf16 on L4/A100, AWQ on a T4
    print(f"{CONFIG}: serving as {SERVED}, weights chosen by GPU")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh

/content/senior-data-scientist
qwen2.5-vl-7b
configs/colab_vllm.yaml: serving as qwen2.5-vl-7b, weights chosen by GPU
GPU 23034 MB, compute 89: serving Qwen/Qwen2.5-VL-7B-Instruct as qwen2.5-vl-7b (quant=none dtype=bfloat16 util=0.9 images=4)
vLLM starting (pid 9104), log: vllm.log
vLLM ready


## 8. Check the server responds

In [8]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 200

{"object":"list","data":[{"id":"qwen2.5-vl-7b","object":"model","created":1792599886,"owned_by":"vllm","root":"Qwen/Qwen2.5-VL-7B-Instruct","parent":null,"max_model_len":16384,"permission":[{"id":"mod

## 9. Download the embedding model (bge-m3)

In [9]:
!python scripts/download_models.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0% 0/30 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/191 [00:00<?, ?B/s]           

Fetching 30 files:   3% 1/30 [00:00<00:07,  4.10it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):  22% 191/878 [00:00<00:00, 787B/s]
Reconstructing (incomplete total...):   5% 878/16.7k [00:00<00:20, 787B/s]
Reconstructing (incomplete total...):   5% 878/16.8k [00:00<00:20, 787B/s]
Reconstructing (incomplete total...):   4% 1.00k/23.0k [00:00<00:27, 787B/s]
Reconstructing (incomplete total...):  93% 23.0k/24.6k [00:00<00:02, 787B/s]
Reconstructing (incomplete total...):  16% 24.6k/156k [00:00<02:47, 787B/s] 

Fetching 30 files:  23% 7/30 [00:00<00:01, 22.82it/s]
Reconstructing (incomplete total...):  24% 156k/642k [00:00<00:00, 533kB/s]
Reconstructing (i

## 10. Ingest the reports

In [10]:
!python -m docagent.cli ingest data/reports
!python -m docagent.cli status

Loading weights: 100% 391/391 [00:00<00:00, 2138.09it/s]
pre tokenize: 100% 1/1 [00:00<00:00, 357.85it/s]
Inference Embeddings: 100% 1/1 [00:00<00:00, 45.39it/s]
/content/senior-data-scientist/src/docagent/store.py:46: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(self.collection, field, qm.PayloadSchemaType.KEYWORD)
pre tokenize: 100% 2/2 [00:00<00:00, 284.88it/s]
Inference Embeddings: 100% 2/2 [00:00<00:00,  3.58it/s]
pre tokenize: 100% 4/4 [00:00<00:00, 229.73it/s]
Inference Embeddings: 100% 4/4 [00:01<00:00,  3.98it/s]
ntsb_16-006.pdf: 26 chunks (12 figures)
ntsb_18-061.pdf: 64 chunks (33 figures)
ntsb_24-013.pdf: 6 chunks (5 figures)
collection reports_bge-m3_v1_bf16desc: 96 chunks
  ntsb_16-006: 26
  ntsb_18-061: 64
  ntsb_24-013: 6


## 11. Launch the chat app (public Gradio share link)

In [11]:
import sys
sys.path.insert(0, "/content/senior-data-scientist/src")
sys.path.insert(0, "/content/senior-data-scientist/app")

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=True,
    prevent_thread_lock=True,
    inline=False,
)

url = demo.share_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8f6833042c955352f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Save `store/` (Hugging Face or Drive, per `PERSIST`; skipped when `"none"`)

In [12]:
push_store()

Session-only run: store/ is not saved (it is rebuilt by cell 10 next time).


## 13. Save run outputs, eval results, traces and logs

In [13]:
push_run()

skipped (not found): outputs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Run downloaded to your computer: docagent-run-20261021-161524.zip


## 14. Ask a few questions in the notebook

In [14]:
import sys, time
sys.path.insert(0, "/content/senior-data-scientist/src")
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 43.56it/s]


### What is the fatigue life of a 2024-T3 spar in cycles?

_The reports do not clearly cover this; treat the answer with caution._

The provided sources do not contain information about the fatigue life of a 2024-T3 spar in cycles. The reports focus on examining the fatigue regions in the spar caps of the accident airplane, identifying the fatigue origin areas, and describing the fracture surfaces. There is no data presented regarding the fatigue life of the material in terms of cycles.

**Sources:** ntsb_18-061 p.1; ntsb_18-061 p.7; ntsb_18-061 p.4

<sub>request eb7d965faf00 · routes: report_qa (rule)</sub>

### Why did the left wing main spar fail?

The left wing main spar failed due to fatigue. This conclusion is supported by the observation of fatigue features such as ratchet marks and smooth crack arrest lines on the fracture surface. The fatigue region extends up to 0.126 inches forward from the hole surface and up to 0.115 inches upward from the lower surface. The fatigue origin area is located within the corner radius between the hole LC-1 bore and the main spar lower surface.

**Sources:** ntsb_18-061 p.7

<sub>request 0982a6f95663 · routes: report_qa (rule)</sub>

### What caused the throttle arm to fracture?

The throttle arm fractured due to bending overstress, as indicated by the examination of the fracture surfaces (Figures 4 and 5). There was no evidence of preexisting cracks.

**Sources:** ntsb_24-013 p.6 (Fig. 4); ntsb_24-013 p.6 (Fig. 5)

<sub>request 7e3231e91f97 · routes: report_qa (rule)</sub>

traces: questions-20261021-163450


## 15. Spans of one request (the latest by default)

In [15]:
!python -m docagent.cli trace last

request 7e3231e91f97  succeeded  7.6s  created 2026-10-21T16:35:31.257Z  batch=questions-20261021-163450
  text:     What caused the throttle arm to fracture?
  versions: {"router": "rules+vlm-v1", "drawing": "drawing-geometry-v1", "report_qa": "report-qa-v5", "llm": "qwen2.5-vl-7b"}  config=8e5ad2f26db8
  llm:      1 model calls, tokens 2707 in / 116 out
  outcome:  {"routes": ["report_qa"], "errors": [], "drawings": [], "answer": "The throttle arm fractured due to bending overstress, as indicated by the examination of the fracture surfaces (Figures 4 and 5). There was no evidence of preexisting cracks.", "grounded": true, "citations": ["ntsb_24-013 p6 fig4"...
spans:
  router  0.00s  {"n_attachments": 0, "route": "report_qa", "tier": "rule"}
  report_qa  7.63s  {"has_text": true, "n_images": 0}
    qa.retrieve  0.09s  {"query": "What caused the throttle arm to fracture?", "retriever": "hybrid-rrf-v3-textfirst-reg-focus-next...
    qa.generate  7.47s  {"n_sources": 11, "n_images": 2, 

## 16. Trace of the latest request

---



In [16]:
import sqlite3, pandas as pd

con = sqlite3.connect(PROJECT / "store/traces.db")
REQUEST_ID = pd.read_sql("select request_id from requests order by created_at desc limit 1", con).iloc[0, 0]
# REQUEST_ID = "53b8c2c82def"   # or any id from a chat reply / `docagent requests`
print("request", REQUEST_ID)
pd.read_sql("select * from spans where request_id = ? order by started_at", con, params=(REQUEST_ID,))

request 7e3231e91f97


,span_id,parent_id,request_id,task_id,name,status,started_at,ended_at,duration_ms,attrs,error
0,2ea1e5561b00,None,7e3231e91f97,None,router,succeeded,2026-10-21T16:35:31.258Z,2026-10-21T16:35:31.258Z,0.134159,"{""n_attachments"": 0, ""route"": ""report_qa"", ""ti...",None
1,12c20bf6a681,None,7e3231e91f97,None,report_qa,succeeded,2026-10-21T16:35:31.258Z,2026-10-21T16:35:38.890Z,7632.234263,"{""has_text"": true, ""n_images"": 0}",None
2,d1dcf2b90d1b,12c20bf6a681,7e3231e91f97,None,qa.retrieve,succeeded,2026-10-21T16:35:31.258Z,2026-10-21T16:35:31.346Z,87.682029,"{""query"": ""What caused the throttle arm to fra...",None
3,3a9aede1ed94,12c20bf6a681,7e3231e91f97,None,qa.generate,succeeded,2026-10-21T16:35:31.418Z,2026-10-21T16:35:38.889Z,7470.477217,"{""n_sources"": 11, ""n_images"": 2, ""n_user_image...",None
4,246e31f6389d,3a9aede1ed94,7e3231e91f97,None,llm.call,succeeded,2026-10-21T16:35:31.499Z,2026-10-21T16:35:38.889Z,7389.299888,"{""model"": ""qwen2.5-vl-7b"", ""n_images"": 2, ""pro...",None
5,80c24ead9f48,12c20bf6a681,7e3231e91f97,None,qa.verify_citations,succeeded,2026-10-21T16:35:38.889Z,2026-10-21T16:35:38.890Z,0.818308,"{""model_cited"": [5, 6], ""kept"": [5, 6]}",None


## 17. Evaluate the model in memory

The next cell finds the model vLLM is serving and its config, and compares each run with the latest pinned run of the
same model in `eval/results/`. The eval runs inside this kernel, next to the chat app (see the cell).

In [17]:
import json, os, subprocess, sys, urllib.request, yaml
from pathlib import Path

PROJECT = Path("/content/senior-data-scientist")
LABEL = None              # None: <awq|bf16>-<gpu>, e.g. bf16-l4: the tag stored with each run
BASELINES = Path("eval/results")   # pinned earlier runs; the latest one of the same model is compared

# Rerunnable, and safe next to the chat app:
# - self-contained: after "Restart session" the kernel forgets CONFIG and the working directory, but vLLM keeps running
# - the eval runs inside this kernel, not as `!python -m eval`: embedded Qdrant (store/qdrant) allows one process,
#   and once the chat app or step 14 has asked a report question, this kernel holds it. In here the eval reuses
#   that open index and the loaded bge-m3 instead of failing with "already accessed by another instance".
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))          # the project's eval/ package (only docagent is pip-installed)
from eval.__main__ import main as eval_main

try:
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as r:
        served = [m["id"] for m in json.load(r)["data"]]
except OSError:
    raise RuntimeError("vLLM is not running: run step 7 first") from None
llm = {c: yaml.safe_load(c.read_text())["llm"] for c in sorted(Path("configs").glob("*.yaml"))}
matches = [c for c, l in llm.items() if l.get("backend") == "vllm" and l["model"] in served]
if len(matches) != 1:
    raise RuntimeError(f"vLLM serves {served}; configs asking for it: {[str(c) for c in matches] or 'none'}")
EVAL_CONFIG = matches[0]                      # the config of the model in memory
MODEL = llm[EVAL_CONFIG]["model"]
if LABEL is None:
    try:                                      # "NVIDIA L4" -> l4, "Tesla T4" -> t4
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.split("\n")[0].split()[-1].lower()
    except (OSError, IndexError):
        gpu = "gpu"
    LABEL = ("awq" if MODEL.endswith("-awq") else "bf16") + "-" + gpu
print(f"evaluating {EVAL_CONFIG} ({MODEL}) as {LABEL!r}")

def baseline(suite):
    """Latest pinned run of this suite with the model in memory, on all cases and with --fresh (as run_eval runs)."""
    found = []
    for p in BASELINES.rglob(f"*_{suite}_*.json"):
        try:
            run = json.loads(p.read_text())["run"]
        except (OSError, ValueError, KeyError, TypeError):
            continue
        if run.get("suite") == suite and run.get("model") == MODEL and run.get("split") == "all" and run.get("fresh"):
            found.append((run.get("timestamp", ""), p))
    return max(found)[1] if found else None

def run_eval(suite, extra="", compare=None):
    """One suite (router | qa_text | qa_image | drawing | e2e) against the model in memory. Rerun freely.
    compare: None = latest pinned run of the same model; a results file (path, or a name found under eval/results/);
    False = no comparison."""
    if compare is None:
        base = baseline(suite)
    elif compare is False:
        base = None
    else:
        base = Path(compare)
        if not base.is_file():                # a bare file name: look it up under eval/results/
            hits = sorted(Path("eval/results").rglob(base.name))
            if not hits:
                raise FileNotFoundError(f"no results file {compare!r} under eval/results/")
            base = hits[0]
        run = json.loads(base.read_text())["run"]
        if run.get("suite") != suite:
            raise ValueError(f"{base} is a {run.get('suite')!r} run, not {suite!r}")
        if run.get("model") != MODEL:
            print(f"note: {base.name} used {run.get('model')}, this run uses {MODEL}")
    print(f"{suite}: baseline {base or ('none' if compare is False else f'none for {MODEL}')}")
    argv = [suite, "--config", str(EVAL_CONFIG), "--split", "all", "--fresh", "--label", LABEL, *extra.split()]
    if base:
        argv += ["--compare", str(base)]
    try:
        eval_main(argv)
    except SystemExit as e:                   # argparse errors / --help: report, don't kill the kernel's state
        if e.code not in (0, None):
            print(f"eval stopped: {e}")

evaluating configs/colab_vllm.yaml (qwen2.5-vl-7b) as 'bf16-l4'


In [18]:
# Rerun freely, with this or another suite: router | qa_text | qa_image | drawing | e2e
# (extra="--n 5" for a quick sample). The chat app can stay up; chats during a run only skew its latency numbers.
# Compare with a specific run: compare="20261019-132202_qa_text_awq-l4.json" (or a path); compare=False: none.

run_eval("qa_image", compare="20261021-163539_qa_image_bf16-l4.json")

qa_image: baseline eval/results/20261019-105306_qa_image_baseline-l4.json



id                     split input      kind           pass  doc  fig  facts  acts  filler  problem
img-24-013-f2          dev   image+text figure         ✗     ✗    ✗    ✗      ✓             missing ['overstress']
img-24-013-f3          test  image      figure         ✗     ✓    ✓    ✗      ✓             forbidden ['corrosion']
img-24-013-f4          dev   image      figure         ✗     ✗    ✗    ✗      ✓             forbidden ['corrosion']
img-16-006-f1          test  image      figure         ✓     ✓    ✓    ✓      ✓             
img-16-006-f3          dev   image      figure         ✓     ✓    ✓    ✓      -             
img-16-006-f7          test  image      figure         ✗     ✓    ✓    ✗      -             missing ['inspection hole']
img-16-006-f9          dev   image      figure         ✓     ✓    ✓    ✓      -             
img-16-006-f12         test  image+text figure         ✓     ✓    ✓    ✓      ✓             
img-18-061-f2          dev   image      figure         ✓    

In [19]:
run_eval("e2e", compare="20261021-165057_e2e_bf16-l4.json")

e2e: baseline eval/results/20261019-113957_e2e_baseline-l4.json



id   split ok  route  parts   what / problem
e01  dev   ✓   ✓      ✓       text only
e02  dev   ✓   ✓      ✓       text only, multi-aircraft
e03  dev   ✓   ✓      ✓       photo only
e04  dev   ✓   ✓      ✓       photo + question
e05  dev   ✓   ✓      ✓       drawing PDF
e06  dev   ✓   ✓      ✓       drawing as a picture
e07  dev   ✓   ✓      ✓       several drawings
e08  dev   ✓   ✓      ✓       text + drawing: current behaviour = drawing JSON only, the question is not answered
e09  test  ✓   ✓      ✓       text only, multi-aircraft
e10  test  ✗   ✓      ✗       text only, must refuse  <- answer failed refusal
e11  test  ✗   ✓      ✗       photo only  <- answer failed facts/doc_hit (missing ['LC-1'])
e12  test  ✓   ✓      ✓       photo + question
e13  test  ✓   ✓      ✓       drawing PDF (unverified GT)
e14  test  ✓   ✓      ✓       drawing PDF, 2 sheets (unverified GT)
e15  test  ✓   ✓      ✓       drawing PDF without hand labels: agreement with the fix4 reference
e16  test  ✗   ✓   

In [20]:
run_eval("qa_text", compare="20261021-171532_qa_text_bf16-l4.json")

qa_text: baseline eval/results/20261019-104621_qa_text_baseline-l4.json

id    split pass  facts  pages  filler   secs  problem
q01   dev   ✓     ✓      ✓                 10  
q02   dev   ✓     ✓      ✓                 10  
q03   dev   ✓     ✓      ✓                  7  
q04   dev   ✓     ✓      ✓                 10  
q05   dev   ✓     ✓      ✓                  9  
q06   dev   ✓     ✓      ✓                 11  
q07   dev   ✓     ✓      ✓                 10  
q08   dev   ✓     ✓      ✓                  9  
q09   dev   ✓     ✓      ✓                 15  
q10   dev   ✓     ✓      ✓                 13  
q11   dev   ✓     ✓      ✓                 18  
q12   dev   ✓     ✓      ✓                 14  
q13   dev   ✓     ✓      ✓                  8  
q14   dev   ✓     ✓      ✓                 19  
q15   dev   ✓     ✓      ✓                 12  
q16   dev   ✗     ✗      ✓                 16  missing ['roll groove']
q17   dev   ✓     ✓      ✓                  9  
q18   dev   ✗     ✗      ✓       

## Reading saved runs back

On Hugging Face, fetch any earlier run, or the store as it was at any commit (see the repo's *Files → History*):
```python
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} --repo-type dataset --include "runs/<RUN_ID>/*" --local-dir /content/saved
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} store-backup.tar.gz --repo-type dataset --revision <commit> --local-dir /content
```
With `"none"` each run is the downloaded `docagent-run-<RUN_ID>.zip`. On Drive they are in `MyDrive/ColabProjects/run_out/<RUN_ID>/`.